# OET Data Transformation Pipeline

A basic data pipeline that:

* Downloads and extracts the OET biomedical dataset to `./data/OET_data`.
* Loads OOKB mentions from the `test-NIL-all.json` files.
* Drops unnessesary columns (e.g., context, edge labels, UMLS concept ids, etc.)
* Converts mentions to lower case
* Removes punctuation from mentions
* Removes short mentions (e.g., accronyms)
* Drops duplicate rows
* Drops rows with complex targets
* Drops rows where targets are within 4 hops of owl:Thing
* Prepares data so it can be further processed by tooling from HR-OOV
* Writes the prepared data to disk

In [35]:
from pathlib import Path
import json
import pandas as pd
from copy import deepcopy
from IPython.display import display, HTML
import re

In [36]:
Path("./data").mkdir(parents=True, exist_ok=True)

In [3]:
! wget https://zenodo.org/records/10432003/files/OET-data-ver4.zip

--2026-01-04 22:50:00--  https://zenodo.org/records/10432003/files/OET-data-ver4.zip
Resolving zenodo.org (zenodo.org)... 188.185.43.153, 188.185.48.75, 137.138.52.235, ...
Connecting to zenodo.org (zenodo.org)|188.185.43.153|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 595373656 (568M) [application/octet-stream]
Saving to: ‘OET-data-ver4.zip’

OET-data-ver4.zip   100%[===================>] 567.79M  14.3MB/s    in 78s     

2026-01-04 22:51:19 (7.24 MB/s) - ‘OET-data-ver4.zip’ saved [595373656/595373656]



In [5]:
! mv OET-data-ver4.zip ./data/.

In [7]:
! unzip -d ./data/OET_data ./data/OET-data-ver4.zip

Archive:  ./data/OET-data-ver4.zip
  inflating: ./data/OET_data/readme.md  
  inflating: ./data/OET_data/__MACOSX/._readme.md  
   creating: ./data/OET_data/MM-S14-Disease/
  inflating: ./data/OET_data/__MACOSX/._MM-S14-Disease  
  inflating: ./data/OET_data/MM-S14-Disease/.DS_Store  
  inflating: ./data/OET_data/__MACOSX/MM-S14-Disease/._.DS_Store  
   creating: ./data/OET_data/MM-S14-Disease/mention-edge-pair-level/
   creating: ./data/OET_data/MM-S14-Disease/mention-level-(out-of-KB-discovery)/
   creating: ./data/OET_data/MM-S14-Disease/mention-level-(concept-placement)/
   creating: ./data/OET_data/MM-S14-Disease/ontology/
  inflating: ./data/OET_data/MM-S14-Disease/mention-edge-pair-level/train.jsonl  
  inflating: ./data/OET_data/__MACOSX/MM-S14-Disease/mention-edge-pair-level/._train.jsonl  
  inflating: ./data/OET_data/MM-S14-Disease/mention-edge-pair-level/test-in-KB.jsonl  
  inflating: ./data/OET_data/__MACOSX/MM-S14-Disease/mention-edge-pair-level/._test-in-KB.jsonl  
  in

In [10]:
! rm -rf ./data/OET_data/__MACOSX

In [37]:
_regex_parens = re.compile(r"\s*\([^)]*\)") # for parentheses removal

In [38]:
def display_data(data, lower_bound=0, upper_bound=1000):
  display(HTML(data[lower_bound:upper_bound].sort_values('mention').to_html()))

In [39]:
def strip_parens(s: str) -> str:
  return _regex_parens.sub("", s)

def get_value_for_key(arr: dict, key: str):
  return arr[key]

In [40]:
def read_jsonl(filepath: Path) -> list[dict]:
  """Read a JSONL file and return a list of dictionaries"""
  records = []
  with filepath.open("r", encoding="utf-8-sig") as f:
    for line in f:
      line = line.strip()
      if line:  # skip empty lines
        records.append(json.loads(line))
  return records

In [41]:
src_root_path = Path('./data/OET_data')

In [42]:
src_cpp_data_path = (src_root_path / 'MM-S14-CPP' / 'mention-edge-pair-level' / 'test-NIL-all.jsonl').expanduser().resolve()

In [43]:
src_disease_data_path = (src_root_path / 'MM-S14-Disease' / 'mention-edge-pair-level' / 'test-NIL-all.jsonl').expanduser().resolve()

In [44]:
cpp_data = read_jsonl(src_cpp_data_path)
print(f"Loaded {len(cpp_data)} records")

Loaded 2131 records


In [45]:
disease_data = read_jsonl(src_disease_data_path)
print(f"Loaded {len(disease_data)} records")

Loaded 1637 records


In [46]:
RENDER_DATA = False

In [47]:
disease_df = pd.DataFrame(disease_data)

if RENDER_DATA:
  display_data(disease_df)

In [48]:
# disease_df

In [49]:
# drop data that we do not require

disease_df = disease_df.drop(columns=['context_left', 'context_right', 'child', 'edge_label_id', 'entity_label_title', 'entity_label', 'child_concept', 'entity_label_id', 'label_concept_ori', 'label_concept', 'label_concept_UMLS'])

if RENDER_DATA:
  display_data(disease_df)

In [50]:
# disease_df

In [51]:
# cast mentions to lower case

disease_df['mention'] = disease_df['mention'].apply(lambda x: x.lower())

if RENDER_DATA:
  display_data(disease_df)

In [52]:
# remove special characters

disease_df['mention'] = disease_df['mention'].replace(r'[^\w\s]', '', regex=True)

if RENDER_DATA:
  display_data(disease_df)

In [53]:
# remove any short mentions (i.e., accronyms, etc.)

disease_df = disease_df[disease_df["mention"].str.len() > 5]

if RENDER_DATA:
  display_data(disease_df)

In [54]:
# drop duplicate rows

disease_df = disease_df.drop_duplicates()

if RENDER_DATA:
  display_data(disease_df)

In [55]:
# drop complex concepts

disease_df = disease_df[pd.to_numeric(disease_df['parent_concept'], errors='coerce').notna()]

if RENDER_DATA:
  display_data(disease_df)

In [56]:
# display_data(disease_df)

In [57]:
# a list of (broad, high level) concepts that sit within 4 hops of owl:Thing

broad_high_level_concepts = ['disease (disorder)', 'procedure (procedure)', 'pharmaceutical / biologic product (product)', 'clinical finding (finding)']

In [58]:
# remove rows with annotated parents that belong to the set of 'broad, high level' concepts

# Note that this allows us to evaluate using d = {1, 3, 5}

# Alternatively, we could skip this step to implement the approach suggested by Hui, i.e., use original and filtered ranking approach
# and entirely disregard d = {1, 3, 5}

# Would this make it difficult to compare with the results from HR-OOV though?

disease_df = disease_df[~disease_df['parent'].isin(broad_high_level_concepts)]

if RENDER_DATA:
  display_data(disease_df)

In [59]:
view_data = True

if view_data:
    print(f"Final dataframe shape: {disease_df.shape}")
    display_data(disease_df)

Final dataframe shape: (80, 3)


,mention,parent_concept,parent
354,alternating skew deviation,40631009,skew deviation (disorder)
243,artery dissections,359557001,disorder of artery (disorder)
289,asthma exacerbation,281239006,exacerbation of asthma (disorder)
978,atypical parkinsonism,32798002,parkinsonism (disorder)
305,cervical spinal cord compression,48522003,spinal cord disorder (disorder)
170,cervical spondylotic myelopathy,48522003,spinal cord disorder (disorder)
2,chronic kidney disease,236423003,renal impairment (disorder)
1307,chronic renal disease,236423003,renal impairment (disorder)
1306,chronic renal insufficiency,236423003,renal impairment (disorder)
390,ckdmbd,236423003,renal impairment (disorder)


In [60]:
disease_record_dicts = disease_df.to_dict('records')

In [61]:
dataset_version = '2026-01'
source_dataset = 'OET-DISEASE-mention-edge-pair-level__TEST-NIL-ALL'
placeholder_context_left = 'NO_CONTEXT_LEFT'
placeholder_context_right = 'NO_CONTEXT_RIGHT'

save_file_name = f'manually-annotated-v{dataset_version}_NIL_TEST_ALL_DISEASE.json'

processed_records = []

for i, record in enumerate(disease_record_dicts):
  processed_records.append({
    'internal_id': f'CPP-V{dataset_version}-{i}',
    'source_dataset': 'OET-CPP-mention-edge-pair-level__TEST-NIL-ALL-SET',
    'question_id': i,
    'question': f"{placeholder_context_left} {get_value_for_key(record, 'mention')} {placeholder_context_right}",
    'entity_mention': {
      'entity_literal': get_value_for_key(record, 'mention')
    },
    'target_entity': {
      'iri': f"http://snomed.info/id/{get_value_for_key(record, 'parent_concept')}",
      'rdfs:label': get_value_for_key(record, 'parent'),
      'skos:prefLabel': strip_parens(get_value_for_key(record, 'parent')),
      'skos:altLabels': [],
      'depth': 0
    }
  })

print(f"Finishing processing {len(processed_records)} records.")

Finishing processing 80 records.


In [62]:
with open(f"./data/{save_file_name}", "w", encoding='utf-8') as file:
  json.dump(processed_records, file, indent=4)